# SOLUCIÓN PROFESORADO – EXAMEN 90 min
## Clasificación: Logística vs KNN vs SVM
Dataset: Driver_Behavior.csv



In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report, roc_curve, roc_auc_score

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

import matplotlib.pyplot as plt


## Columnas
- speed_kmph
- accel_x,
- accel_y,
- brake_pressure,
- steering_angle,
- throttle,
- lane_deviation,
- phone_usage,
- headway_distance,
- reaction_time,
- behavior_label

In [2]:
df = pd.read_csv("Driver_Behavior.csv")

display(df.head())
print("Shape:", df.shape)
df.info()


FileNotFoundError: [Errno 2] No such file or directory: 'Driver_Behavior.csv'

In [ ]:
# Variable objetivo
y = df["behavior_label"]
X = df.drop(columns=["behavior_label"])

print("Distribución de clases:")
print(y.value_counts())

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Escalado
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
def evaluar(modelo, X_test, y_test, nombre):
    y_pred = modelo.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    
    print(f"\n=== {nombre} ===")
    print("Accuracy:", round(acc, 4))
    print("Matriz de confusión:\n", confusion_matrix(y_test, y_pred))
    print("Classification report:\n", classification_report(y_test, y_pred))
    
    try:
        scores = modelo.predict_proba(X_test)[:, 1]
        auc = roc_auc_score(y_test, scores)
        fpr, tpr, _ = roc_curve(y_test, scores)
        plt.figure()
        plt.plot(fpr, tpr)
        plt.plot([0,1],[0,1],'--')
        plt.title(f"ROC - {nombre} (AUC={auc:.3f})")
        plt.xlabel("FPR")
        plt.ylabel("TPR")
        plt.show()
        return acc, auc
    except:
        return acc, None


In [3]:
log_reg = LogisticRegression(max_iter=2000, random_state=42)
log_reg.fit(X_train_scaled, y_train)

acc_log, auc_log = evaluar(log_reg, X_test_scaled, y_test, "Regresión Logística")


NameError: name 'X_train_scaled' is not defined

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

acc_knn5, auc_knn5 = evaluar(knn, X_test_scaled, y_test, "KNN (k=5)")

# Búsqueda de k
k_values = [1, 5, 11, 21]
accs = []

for k in k_values:
    m = KNeighborsClassifier(n_neighbors=k)
    m.fit(X_train_scaled, y_train)
    y_pred = m.predict(X_test_scaled)
    accs.append(accuracy_score(y_test, y_pred))

plt.figure()
plt.plot(k_values, accs, marker="o")
plt.title("Accuracy vs k")
plt.xlabel("k")
plt.ylabel("Accuracy")
plt.show()


In [ ]:
svm_lin = SVC(kernel="linear", C=1.0, probability=True, random_state=42)
svm_lin.fit(X_train_scaled, y_train)
acc_svm_lin, auc_svm_lin = evaluar(svm_lin, X_test_scaled, y_test, "SVM Lineal")

svm_rbf = SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=42)
svm_rbf.fit(X_train_scaled, y_train)
acc_svm_rbf, auc_svm_rbf = evaluar(svm_rbf, X_test_scaled, y_test, "SVM RBF")


In [ ]:
resumen = pd.DataFrame([
    {"Modelo": "Logística", "Accuracy": acc_log, "AUC": auc_log},
    {"Modelo": "KNN (k=5)", "Accuracy": acc_knn5, "AUC": auc_knn5},
    {"Modelo": "SVM lineal", "Accuracy": acc_svm_lin, "AUC": auc_svm_lin},
    {"Modelo": "SVM RBF", "Accuracy": acc_svm_rbf, "AUC": auc_svm_rbf},
])

display(resumen)



## Conclusión modelo recomendado (ejemplo)

En general, SVM RBF suele capturar mejor relaciones no lineales entre variables como `lane_deviation`, `reaction_time` y `accel_x`.  
En problemas de seguridad vial es especialmente relevante maximizar el recall de la clase de riesgo para minimizar falsos negativos.
